In [11]:
import os

# def open_whatsapp():
#     os.system(
#         r'start "" "shell:AppsFolder\5319275A.WhatsAppDesktop_cv1g1gvanyjgm!App"'
#     )

In [12]:
import math
import subprocess
import time

import cv2
import mediapipe as mp
import pyautogui



In [ ]:
"""Dual-hand gesture controller for Windows.
Right hand: mouse, click, scroll and screenshot.
Left hand: volume and application launcher.
"""



MOUSE_HAND = "Right"
CONTROL_HAND = "Left"

APP_PATHS = {
    "Chrome": r"C:\Program Files\Google\Chrome\Application\chrome.exe",
    "Edge": r"C:\Program Files (x86)\Microsoft\Edge\Application\msedge.exe",
    "Excel" : r"C:\Program Files\Microsoft Office\root\Office16\EXCEL.EXE",  
    # "VSCode": r"C:\Users\Vedant Singh\AppData\Local\Programs\Microsoft VS Code\Code.exe", 
}


def finger_count(landmarks):
    """Return how many of index, middle, ring and little fingers are raised."""
    return sum(
        landmarks.landmark[tip].y < landmarks.landmark[tip - 2].y
        for tip in (8, 12, 16, 20)
    )


def launch_app(app_name):
    target = APP_PATHS[app_name]
    try:
        if target.endswith(":"):
            os.startfile(target)
        elif os.path.exists(target):
            subprocess.Popen([target])
        else:
            raise FileNotFoundError(target)
        return True
    except OSError as error:
        print(f"Could not open {app_name}: {error}")
        return False




In [ ]:
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7,
)

In [ ]:


cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
if not cap.isOpened():
    raise RuntimeError("Camera could not be opened. Try another camera index.")

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_FPS, 30)

screen_w, screen_h = pyautogui.size()
click_latched = False
click_times = []
last_screenshot_time = 0.0
last_volume_time = 0.0
last_launcher_time = 0.0
launcher_latched = False

VOLUME_DELAY = 0.08
LAUNCH_COOLDOWN = 2.5
SCREENSHOT_COOLDOWN = 4.0

cv2.namedWindow("Dual Hand Gesture Control", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Dual Hand Gesture Control", 700, 520)

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        frame = cv2.flip(frame, 1)
        result = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        status = "Show both hands"

        if result.multi_hand_landmarks and result.multi_handedness:
            for hand_landmarks, handedness in zip(
                result.multi_hand_landmarks, result.multi_handedness
            ):
                label = handedness.classification[0].label
                mp_drawing.draw_landmarks(
                    frame, hand_landmarks, mp_hands.HAND_CONNECTIONS
                )
                tip_thumb = hand_landmarks.landmark[4]
                tip_index = hand_landmarks.landmark[8]
                raised = finger_count(hand_landmarks)
                pinch_distance = math.hypot(
                    tip_thumb.x - tip_index.x, tip_thumb.y - tip_index.y
                )

                # RIGHT HAND: original mouse features.
                if label == MOUSE_HAND:
                    if pinch_distance < 0.06:
                        if not click_latched:
                            click_latched = True
                            click_times.append(time.time())
                            if len(click_times) >= 2 and click_times[-1] - click_times[-2] < 0.4:
                                pyautogui.doubleClick()
                                click_times.clear()
                                status = "Mouse: double click"
                            else:
                                pyautogui.click()
                                status = "Mouse: click"
                    else:
                        click_latched = False
                        pyautogui.moveTo(
                            int(tip_index.x * screen_w),
                            int(tip_index.y * screen_h),
                            duration=0.03,
                        )

                    if raised == 4:  # Four raised fingers = scroll mode.
                        if tip_index.y < 0.30:
                            pyautogui.scroll(8)
                            status = "Mouse: scroll up"
                        elif tip_index.y > 0.55:
                            pyautogui.scroll(-8)
                            status = "Mouse: scroll down"
                    elif raised == 0 and time.time() - last_screenshot_time > SCREENSHOT_COOLDOWN:
                        pyautogui.screenshot(f"screenshot_{int(time.time())}.png")
                        last_screenshot_time = time.time()
                        status = "Mouse: screenshot saved"

                # LEFT HAND: four fingers = volume from thumb-index distance.
                elif label == CONTROL_HAND:
                    if raised == 4:
                        # Small gap lowers volume; wide gap raises it.
                        now = time.time()
                        if now - last_volume_time > VOLUME_DELAY:
                            if pinch_distance > 0.18:
                                pyautogui.press("volumeup")
                            elif pinch_distance < 0.08:
                                pyautogui.press("volumedown")
                            last_volume_time = now
                        status = "Volume: open fingers + pinch gap"
                        launcher_latched = False
                    else:
                        # Hold a sign briefly, then release your hand before the next app.
                        launch_gestures = {1: "Chrome", 2: "Edge", 3: "Excel"}
                        app_name = launch_gestures.get(raised)
                        now = time.time()
                        if app_name and not launcher_latched and now - last_launcher_time > LAUNCH_COOLDOWN:
                            if launch_app(app_name):
                                last_launcher_time = now
                                launcher_latched = True
                                status = f"Opened {app_name}"
                        elif app_name:
                            status = f"Launch gesture: {app_name}"
                        else:
                            launcher_latched = False

        cv2.putText(frame, status, (12, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(frame, "Right: mouse | Left: volume / apps | q: quit", (12, 465), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (255, 255, 255), 1)
        cv2.imshow("Dual Hand Gesture Control", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
finally:
    cap.release()
    hands.close()
    cv2.destroyAllWindows()